# 11 — Structured JSON Generation and Validation

Make predictions machine-valid, schema-consistent, and safe to write as JSONL.

> **Course principle:** every notebook separates **general-purpose knowledge** from **Manga Task application**.

## Define the schema

A schema catches accidental prose, missing fields, wrong page counts, and malformed speaker/text objects.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Utterance(BaseModel):
    speaker: str
    text: str

class SequencePrediction(BaseModel):
    sequence_id: str
    pages: List[List[Utterance]]

pred = SequencePrediction(
    sequence_id="seq_demo",
    pages=[
        [Utterance(speaker="char1", text="Hello.")],
        [],
        [Utterance(speaker="NARRATION", text="Later...")]
    ]
)
pred

## Hard validation rules

For this challenge, add project-specific checks such as exactly three pages and exact narrator spelling.

In [ ]:
def validate_sequence(pred):
    if len(pred.pages) != 3:
        raise ValueError("Each sequence must contain exactly 3 page lists.")

    for page in pred.pages:
        for utt in page:
            if not utt.speaker.strip():
                raise ValueError("Empty speaker.")
            if utt.speaker.lower() == "narration" and utt.speaker != "NARRATION":
                raise ValueError("Narrator must use exact label NARRATION.")

    return True

validate_sequence(pred)

## Writing JSONL

JSONL means one complete JSON object per line.

In [ ]:
import json
from pathlib import Path

out_path = Path("predictions_demo.jsonl")

records = [pred.model_dump()]
with out_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(out_path.read_text(encoding="utf-8"))

## Defensive generation

A robust pipeline can:
1. request JSON only;
2. extract the first valid JSON object if extra text appears;
3. validate with Pydantic;
4. retry locally with a constrained repair prompt if invalid;
5. never manually edit final test outputs.